# Distance-conditioned saturation and mitigation in zero-shot cross-lingual transfer

Experiments for the cross-lingual NLI study. Three models from different
families are fine-tuned on English NLI and evaluated zero-shot on XNLI; I then
fit a saturation model relating transfer accuracy to pretraining data and
typological distance, diagnose each language's dominant deficit, and test two
mitigations (a typologically intermediate pivot source vs. added matched data).

Models: `xlm-roberta-base` (encoder), `facebook/mbart-large-50`
(encoder-decoder), `Qwen/Qwen2.5-0.5B` (decoder). Set `QUICK_SMOKE = True` for a
fast pipeline check on tiny subsamples; `False` for the real run.

## Setup

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" \
                "sentencepiece" "scikit-learn" "statsmodels" "lang2vec" 2>/dev/null

In [ ]:
import os, json, math, time, warnings, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed)
from sklearn.metrics import accuracy_score, confusion_matrix
from scipy import stats
from scipy.optimize import curve_fit
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = DEVICE == "cuda" and not USE_BF16
print("device:", DEVICE, "| torch:", torch.__version__,
      "| precision:", "bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32")

## Configuration

In [ ]:
QUICK_SMOKE       = False
XNLI_REPO         = "facebook/xnli"
OUTDIR            = "outputs"
os.makedirs(f"{OUTDIR}/figs", exist_ok=True)

MODELS = {
    "xlmr":  "xlm-roberta-base",
    "mbart": "facebook/mbart-large-50",
    "qwen":  "Qwen/Qwen2.5-0.5B",
}

XNLI_LANGS = ["ar","bg","de","el","en","es","fr","hi","ru","sw","th","tr","ur","vi","zh"]
SOURCE     = "en"
TARGETS    = [l for l in XNLI_LANGS if l != SOURCE]
MITIGATION_TARGETS = ["ur","hi","el","ar","th","sw"]

MAX_LEN           = 128
NUM_EPOCHS        = 2
LR                = 2e-5
TRAIN_SAMPLE_SIZE = 40000
EVAL_SAMPLE_SIZE  = 2000
BATCH = {"xlmr": 32, "mbart": 8, "qwen": 16}
TAU   = 2.0   # knee threshold: accuracy gain (points) per decade of data

if QUICK_SMOKE:
    MODELS = {"xlmr": "xlm-roberta-base"}
    TARGETS = ["ur","hi","el","de"]
    MITIGATION_TARGETS = ["ur","hi"]
    TRAIN_SAMPLE_SIZE, EVAL_SAMPLE_SIZE, NUM_EPOCHS = 800, 300, 1

# HF now requires namespaced dataset ids; route any bare "xnli" call accordingly.
import datasets as _ds
_orig_load_dataset = getattr(_ds, "_orig_load_dataset", _ds.load_dataset)
_ds._orig_load_dataset = _orig_load_dataset
def load_dataset(path, *a, **k):
    if path == "xnli": path = XNLI_REPO
    return _orig_load_dataset(path, *a, **k)
_ds.load_dataset = load_dataset

print("targets:", TARGETS)

## Pretraining sizes and typological distances

`cc_gib`: CC-100 pretraining size (GiB) per language from the XLM-R paper.
`syn_dist`: cosine distance to English on the lang2vec `syntax_knn` vector, with
a fallback table if the package or URIEL is unavailable.

In [ ]:
CC_GIB = {
    "en":300.8,"ru":278.0,"de":66.6,"fr":56.8,"es":53.3,"vi":137.3,"zh":46.9,
    "tr":20.9,"ar":28.0,"el":7.4,"bg":57.5,"hi":20.2,"sw":1.6,"th":71.7,"ur":5.7,
}
ISO3 = {"ar":"ara","bg":"bul","de":"deu","el":"ell","en":"eng","es":"spa",
        "fr":"fra","hi":"hin","ru":"rus","sw":"swa","th":"tha","tr":"tur",
        "ur":"urd","vi":"vie","zh":"cmn"}
SYN_DIST_FALLBACK = {
    "ar":0.63,"bg":0.50,"de":0.39,"el":0.50,"es":0.44,"fr":0.42,"hi":0.60,
    "ru":0.52,"sw":0.61,"th":0.58,"tr":0.62,"ur":0.61,"vi":0.55,"zh":0.58,
}

def get_syntactic_distances(langs):
    try:
        import lang2vec.lang2vec as l2v
        feats = l2v.get_features([ISO3[l] for l in langs]+[ISO3["en"]], "syntax_knn")
        eng = np.array(feats[ISO3["en"]], dtype=float)
        out = {}
        for l in langs:
            v = np.array(feats[ISO3[l]], dtype=float)
            m = ~(np.isnan(v) | np.isnan(eng))
            cos = np.dot(v[m], eng[m])/(np.linalg.norm(v[m])*np.linalg.norm(eng[m]))
            out[l] = float(1.0 - cos)
        return out, "lang2vec"
    except Exception as e:
        print("lang2vec unavailable, using fallback:", type(e).__name__)
        return dict(SYN_DIST_FALLBACK), "fallback"

SYN_DIST, dist_src = get_syntactic_distances([l for l in XNLI_LANGS if l!="en"])
meta = pd.DataFrame({"lang": TARGETS})
meta["cc_gib"]   = meta.lang.map(CC_GIB)
meta["log_cc"]   = np.log10(meta.cc_gib)
meta["syn_dist"] = meta.lang.map(SYN_DIST)
print("distance source:", dist_src)
meta

## Pivot selection

For each mitigation target, the pivot is the XNLI language (excluding English
and the target) with the smallest syntactic distance to the target and closer to
it than English.

In [ ]:
def syn_dist_between(a, b):
    try:
        import lang2vec.lang2vec as l2v
        feats = l2v.get_features([ISO3[a], ISO3[b]], "syntax_knn")
        va, vb = np.array(feats[ISO3[a]],float), np.array(feats[ISO3[b]],float)
        m = ~(np.isnan(va)|np.isnan(vb))
        cos = np.dot(va[m],vb[m])/(np.linalg.norm(va[m])*np.linalg.norm(vb[m]))
        return float(1-cos)
    except Exception:
        return abs(SYN_DIST[a]-SYN_DIST[b])

def choose_pivot(target):
    cands = [l for l in XNLI_LANGS if l not in (target,"en")]
    scored = sorted(cands, key=lambda p: syn_dist_between(p, target))
    for p in scored:
        if syn_dist_between(p, target) < SYN_DIST[target]:
            return p, syn_dist_between(p, target)
    return scored[0], syn_dist_between(scored[0], target)

PIVOTS = {t: choose_pivot(t) for t in MITIGATION_TARGETS}
for t,(p,d) in PIVOTS.items():
    print(f"{t} -> pivot {p} (dist {d:.2f}); english dist {SYN_DIST[t]:.2f}")

## Data and training helpers

XNLI labels are shared across configs: 0 entailment, 1 neutral, 2 contradiction.
Every language config has a (machine-translated for non-English) train split and
a human test split, so the pivot and matched-data arms reuse the same loader.

In [ ]:
def load_nli_train(lang, n):
    ds = load_dataset(XNLI_REPO, lang, split="train")
    return ds.shuffle(seed=SEED).select(range(n)) if n and n < len(ds) else ds

def load_nli_test(lang, n):
    ds = load_dataset(XNLI_REPO, lang, split="test")
    return ds.shuffle(seed=SEED).select(range(n)) if n and n < len(ds) else ds

def make_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def tokenize_ds(ds, tok):
    def f(b): return tok(b["premise"], b["hypothesis"], truncation=True, max_length=MAX_LEN)
    cols = [c for c in ds.column_names if c != "label"]
    return ds.map(f, batched=True, remove_columns=cols)

def build_model(model_name):
    m = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
    if m.config.pad_token_id is None:
        m.config.pad_token_id = make_tokenizer(model_name).pad_token_id
    return m

def fine_tune(model_name, train_ds, tag):
    tok = make_tokenizer(model_name)
    model = build_model(model_name)
    args = TrainingArguments(
        output_dir=f"{OUTDIR}/ckpt_{tag}",
        per_device_train_batch_size=BATCH.get(tag.split("_")[0], 16),
        gradient_accumulation_steps=2, num_train_epochs=NUM_EPOCHS,
        learning_rate=LR, fp16=USE_FP16, bf16=USE_BF16, logging_steps=50,
        save_strategy="no", report_to="none", seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=tokenize_ds(train_ds, tok),
                      data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    return trainer, tok

def evaluate_on(trainer, tok, lang, n):
    test = load_nli_test(lang, n)
    pred = trainer.predict(tokenize_ds(test, tok))
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    yhat = logits.argmax(-1)
    y = np.array(test["label"])
    return accuracy_score(y, yhat)*100, y, yhat, test

def free():
    if DEVICE == "cuda": torch.cuda.empty_cache()

## Zero-shot baseline (English source)

Standard zero-shot protocol and the baseline every mitigation is measured
against. Results cached to CSV.

In [ ]:
zs_path = f"{OUTDIR}/results_zeroshot.csv"
rows = []
for tag, mname in MODELS.items():
    print(f"\n[{tag}] fine-tuning on English NLI")
    t0 = time.time()
    trainer, tok = fine_tune(mname, load_nli_train(SOURCE, TRAIN_SAMPLE_SIZE), f"{tag}_en")
    print(f"  trained in {time.time()-t0:.0f}s")
    for lang in TARGETS:
        acc,_,_,_ = evaluate_on(trainer, tok, lang, EVAL_SAMPLE_SIZE)
        rows.append({"model":tag,"lang":lang,"arm":"baseline","source":"en",
                     "acc":round(acc,2),"cc_gib":CC_GIB[lang],
                     "log_cc":round(np.log10(CC_GIB[lang]),3),
                     "syn_dist":round(SYN_DIST[lang],3)})
        print(f"    {lang}: {acc:5.2f}")
    del trainer; free()

zs = pd.DataFrame(rows); zs.to_csv(zs_path, index=False)
zs.pivot_table(index="lang", columns="model", values="acc")

## Saturation model, knee, and deficit diagnosis

Accuracy is modelled as a concave saturating function of $\log_{10}$ pretraining
data whose ceiling and rate depend on distance, fit on the pooled
(model $\times$ language) points:
$$\mathrm{Acc}=A(d)-B\,e^{-k(d)(\log_{10}D-\ell_0)},\quad A(d)=A_0-A_1 d,\;
k(d)=\max(k_0-k_1 d,0.05).$$
The saturation knee is the $\log_{10}$-data value where the marginal return
drops below `TAU`. Above the knee but still low = distance-limited; below =
data-limited. A linear interaction model with bootstrap CI is reported alongside
as a robustness check.

In [ ]:
zs = pd.read_csv(zs_path)
x, d, y = zs.log_cc.values, zs.syn_dist.values, zs.acc.values
l0 = x.min()

def dcsm(X, A0, A1, B, k0, k1):
    lc, dd = X
    return (A0 - A1*dd) - B*np.exp(-np.maximum(k0-k1*dd, 0.05)*(lc-l0))

popt = None
try:
    popt,_ = curve_fit(dcsm, (x,d), y, p0=[80,20,30,1.0,0.5], maxfev=20000)
    yhat = dcsm((x,d), *popt)
    r2 = 1 - np.sum((y-yhat)**2)/np.sum((y-y.mean())**2)
    A0,A1,B,k0,k1 = popt
    print(f"DCSM: A0={A0:.1f} A1={A1:.1f} B={B:.1f} k0={k0:.2f} k1={k1:.2f} | R^2={r2:.2f}")
except Exception as e:
    print("DCSM fit failed:", e)

zs["c_lcc"] = zs.log_cc - zs.log_cc.mean()
zs["c_d"]   = zs.syn_dist - zs.syn_dist.mean()
mI = smf.ols("acc ~ c_lcc * c_d", data=zs).fit()
b3 = mI.params["c_lcc:c_d"]
boot = []
for _ in range(2000):
    s = zs.sample(len(zs), replace=True)
    try: boot.append(smf.ols("acc ~ c_lcc * c_d", data=s).fit().params["c_lcc:c_d"])
    except Exception: pass
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"interaction beta3 = {b3:+.2f}  95% CI [{lo:+.2f}, {hi:+.2f}]")

def marginal_return(lc, dd):
    A0,A1,B,k0,k1 = popt; k = max(k0-k1*dd, 0.05)
    return B*k*np.exp(-k*(lc-l0))

def knee_logcc(dd, tau=TAU):
    A0,A1,B,k0,k1 = popt; k = max(k0-k1*dd, 0.05)
    val = B*k/tau
    return l0 + (np.log(val)/k if val > 0 else np.inf)

diag = None
if popt is not None:
    recs = []
    for _,r in meta.iterrows():
        knee = knee_logcc(r.syn_dist)
        above = r.log_cc >= knee
        recs.append({"lang":r.lang,"syn_dist":r.syn_dist,"log_cc":round(r.log_cc,2),
                     "knee_log_cc":round(knee,2),"above_knee":bool(above),
                     "marg_return":round(marginal_return(r.log_cc, r.syn_dist),2),
                     "deficit":"distance-limited" if above else "data-limited"})
    diag = pd.DataFrame(recs).sort_values("syn_dist")
    diag.to_csv(f"{OUTDIR}/deficit_diagnosis.csv", index=False)
    print(); print(diag.to_string(index=False))

## Figures

In [ ]:
plt.rcParams.update({"font.family":"serif","savefig.dpi":300})

fig, ax = plt.subplots(figsize=(5,3.6))
sc = ax.scatter(zs.log_cc, zs.acc, c=zs.syn_dist, cmap="viridis",
                s=55, edgecolor="k", linewidth=.4)
mA = smf.ols("acc ~ log_cc", data=zs).fit()
xs = np.linspace(zs.log_cc.min(), zs.log_cc.max(), 100)
ax.plot(xs, mA.params["Intercept"]+mA.params["log_cc"]*xs, "--", color=".3",
        label=f"linear fit $R^2$={mA.rsquared:.2f}")
plt.colorbar(sc, label="syntactic distance from English")
ax.set_xlabel(r"pretraining size, $\log_{10}$ GiB"); ax.set_ylabel("zero-shot XNLI acc")
ax.legend(frameon=False)
for s in ["top","right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUTDIR}/figs/fig_degradation.png", bbox_inches="tight"); plt.show()

if popt is not None:
    fig, ax = plt.subplots(figsize=(5,3.6))
    grid = np.linspace(zs.log_cc.min(), zs.log_cc.max(), 200)
    for dd, lab, col in [(np.percentile(d,15),"close","#1b4965"),
                         (np.percentile(d,85),"distant","#bc4749")]:
        ax.plot(grid, [dcsm((g,dd),*popt) for g in grid], color=col, lw=2, label=lab)
        kn = knee_logcc(dd)
        if np.isfinite(kn) and grid.min() <= kn <= grid.max():
            ax.axvline(kn, color=col, ls=":", lw=1)
    ax.set_xlabel(r"pretraining size, $\log_{10}$ GiB"); ax.set_ylabel("predicted acc")
    ax.legend(frameon=False); ax.set_title(f"Saturation curves (knee at {TAU} pts/decade)")
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.savefig(f"{OUTDIR}/figs/fig_saturation.png", bbox_inches="tight"); plt.show()

## Mitigation experiment

Two arms per target relative to the English-source baseline:
matched-data (English + machine-translated target NLI) and pivot (fine-tune on
the pivot language). Run across all enabled models.

In [ ]:
mit_path = f"{OUTDIR}/results_mitigation.csv"
mrows = []
def add_row(model, lang, arm, source, acc):
    mrows.append({"model":model,"lang":lang,"arm":arm,"source":source,
                  "acc":round(acc,2),"syn_dist":round(SYN_DIST[lang],3)})

for tag, mname in MODELS.items():
    for t in MITIGATION_TARGETS:
        print(f"[{tag}] matched-data: {t}")
        en = load_nli_train(SOURCE, TRAIN_SAMPLE_SIZE//2)
        tg = load_nli_train(t, TRAIN_SAMPLE_SIZE//2)
        trainer, tok = fine_tune(mname, concatenate_datasets([en,tg]).shuffle(seed=SEED),
                                 f"{tag}_match_{t}")
        acc,_,_,_ = evaluate_on(trainer, tok, t, EVAL_SAMPLE_SIZE)
        add_row(tag, t, "matched-data", f"en+{t}", acc); print(f"   {acc:.2f}")
        del trainer; free()
    for t in MITIGATION_TARGETS:
        p,_ = PIVOTS[t]
        print(f"[{tag}] pivot ({p}): {t}")
        trainer, tok = fine_tune(mname, load_nli_train(p, TRAIN_SAMPLE_SIZE), f"{tag}_piv_{t}")
        acc,_,_,_ = evaluate_on(trainer, tok, t, EVAL_SAMPLE_SIZE)
        add_row(tag, t, "pivot", p, acc); print(f"   {acc:.2f}")
        del trainer; free()

mit = pd.DataFrame(mrows); mit.to_csv(mit_path, index=False)
base = zs[zs.arm=="baseline"][["model","lang","acc"]].rename(columns={"acc":"baseline_acc"})
comp = mit.merge(base, on=["model","lang"], how="left")
comp["delta"] = comp.acc - comp.baseline_acc
comp.to_csv(f"{OUTDIR}/results_mitigation_deltas.csv", index=False)
comp.sort_values(["model","lang","arm"])

## Routing vs. fixed policy

Route each target to the mitigation matched to its diagnosed deficit
(distance-limited $\to$ pivot, data-limited $\to$ matched-data) and compare
against the fixed policies and the baseline.

In [ ]:
comp = pd.read_csv(f"{OUTDIR}/results_mitigation_deltas.csv")
try:
    diag_df = pd.read_csv(f"{OUTDIR}/deficit_diagnosis.csv")[["lang","deficit"]]
except Exception:
    diag_df = pd.DataFrame({"lang":MITIGATION_TARGETS,"deficit":"distance-limited"})

summ = []
for tag in comp.model.unique():
    c = comp[comp.model==tag].merge(diag_df, on="lang", how="left")
    base = c.groupby("lang").baseline_acc.first()
    piv  = c[c.arm=="pivot"].set_index("lang").acc
    mat  = c[c.arm=="matched-data"].set_index("lang").acc
    langs = [l for l in MITIGATION_TARGETS if l in piv.index and l in mat.index]
    routed = [piv[l] if c[c.lang==l].deficit.iloc[0]=="distance-limited" else mat[l]
              for l in langs]
    summ.append({"model":tag,
                 "baseline":round(base[langs].mean(),2),
                 "always_pivot":round(piv[langs].mean(),2),
                 "always_matched":round(mat[langs].mean(),2),
                 "routed":round(np.mean(routed),2)})
S = pd.DataFrame(summ); S.to_csv(f"{OUTDIR}/policy_summary.csv", index=False)
print(S.to_string(index=False))

det = comp.merge(diag_df, on="lang", how="left")
pvm = det.pivot_table(index=["model","lang","deficit"], columns="arm",
                      values="acc").reset_index()
pvm["pivot_minus_matched"] = pvm.get("pivot") - pvm.get("matched-data")
print(); print(pvm.to_string(index=False))

## Error analysis

Confusion matrix, per-label accuracy, and sample misclassifications for the
worst baseline target on the primary model.

In [ ]:
primary = list(MODELS)[0]; mname = MODELS[primary]
trainer, tok = fine_tune(mname, load_nli_train(SOURCE, TRAIN_SAMPLE_SIZE), f"{primary}_err")
worst = pd.read_csv(zs_path).query("model==@primary").sort_values("acc").lang.iloc[0]
print("worst baseline target on", primary, ":", worst)

acc, y, yhat, test = evaluate_on(trainer, tok, worst, EVAL_SAMPLE_SIZE)
labels = ["entail","neutral","contra"]
cm = confusion_matrix(y, yhat)
print("\nconfusion matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=labels, columns=labels))
print("\nper-label accuracy:")
for i,l in enumerate(labels):
    print(f"  {l}: {cm[i,i]/cm[i].sum()*100:.1f}%" if cm[i].sum() else f"  {l}: n/a")

print("\nsample misclassifications:")
for i in [j for j in range(len(y)) if y[j]!=yhat[j]][:5]:
    print(f"  P: {test[i]['premise'][:80]}")
    print(f"  H: {test[i]['hypothesis'][:80]}")
    print(f"  true={labels[y[i]]} pred={labels[yhat[i]]}\n")
del trainer; free()